# Kaggle DKT Training Pipeline

> **IMPORTANT KAGGLE SETUP:**
> Configure your Kaggle Notebook settings before running:
> 1. Set Accelerator to **GPU T4x2**.
> 2. Add the Riiid dataset using `Add Data` -> Search "Riiid Answer Correctness Prediction"
>
> *(Note: We use Kaggle's standard input dataset paths in the code below. If you use a custom path locally, please update the input paths appropriately)*.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split


def find_kaggle_train_csv():
    """Find train.csv in Kaggle input mounts."""
    candidates = [
        "/kaggle/input/riiid-answer-correctness-prediction/train.csv",
        "/kaggle/input/riiid-test-answer-prediction/train.csv",
    ]

    for c in candidates:
        if os.path.exists(c):
            return c

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for p in input_root.glob("**/train.csv"):
            return str(p)

    raise FileNotFoundError(
        "Could not find train.csv under /kaggle/input. Add the Riiid dataset in Kaggle first."
    )


def process_data(input_csv=None, output_dir="/kaggle/working/data/processed", max_seq_len=100, max_rows=500000):
    if input_csv is None:
        input_csv = find_kaggle_train_csv()

    print(f"Loading data from {input_csv} (max {max_rows} rows)...")
    df = pd.read_csv(input_csv, nrows=max_rows)

    # Filter out lecture events; keep only question interactions.
    if "content_type_id" in df.columns:
        df = df[df.content_type_id == 0]

    df = df[["user_id", "content_id", "answered_correctly"]].copy()
    df.columns = ["user_id", "question_id", "answered_correctly"]

    print("Encoding question IDs...")
    q_unique = df["question_id"].unique()
    q_mapping = {q: i + 1 for i, q in enumerate(q_unique)}  # 0 reserved for padding
    df["question_id"] = df["question_id"].map(q_mapping)

    print("Grouping into sequences...")
    grouped = df.groupby("user_id").agg({
        "question_id": list,
        "answered_correctly": list,
    })

    sequences_q = [torch.tensor(q, dtype=torch.long) for q in grouped["question_id"]]
    sequences_a = [torch.tensor(a, dtype=torch.float32) for a in grouped["answered_correctly"]]

    print(f"Padding sequences to length {max_seq_len}...")
    sequences_q = [q[-max_seq_len:] for q in sequences_q]
    sequences_a = [a[-max_seq_len:] for a in sequences_a]

    padded_q = pad_sequence(sequences_q, batch_first=True, padding_value=0)
    padded_a = pad_sequence(sequences_a, batch_first=True, padding_value=-1)

    # Force exact width for consistent training tensor shapes.
    if padded_q.size(1) < max_seq_len:
        pad_size = max_seq_len - padded_q.size(1)
        padded_q = torch.cat([padded_q, torch.zeros(padded_q.size(0), pad_size, dtype=torch.long)], dim=1)
        padded_a = torch.cat([padded_a, torch.full((padded_a.size(0), pad_size), -1, dtype=torch.float32)], dim=1)

    print("Splitting dataset 80/10/10...")
    indices = list(range(padded_q.size(0)))
    train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

    print("Saving tensors...")
    os.makedirs(output_dir, exist_ok=True)
    torch.save({"q": padded_q[train_idx], "a": padded_a[train_idx]}, os.path.join(output_dir, "train.pt"))
    torch.save({"q": padded_q[val_idx], "a": padded_a[val_idx]}, os.path.join(output_dir, "val.pt"))
    torch.save({"q": padded_q[test_idx], "a": padded_a[test_idx]}, os.path.join(output_dir, "test.pt"))

    print(
        f"Preprocessing complete. Saved {len(train_idx)} train, {len(val_idx)} val, {len(test_idx)} test sequences."
    )
    print(f"Unique questions encoded: {len(q_unique)}")


# Run preprocessing in Kaggle
process_data()

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score


class DKT(nn.Module):
    def __init__(self, n_questions, embed_size=128, hidden_size=128, num_layers=2):
        super().__init__()
        self.n_questions = n_questions
        self.embedding = nn.Embedding(2 * n_questions + 1, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_questions + 1)

    def forward(self, q_seq, a_seq):
        # interaction_id = question_id + n_questions * correctness
        interaction = q_seq + self.n_questions * a_seq.clamp(min=0).long()
        interaction = interaction.masked_fill(q_seq == 0, 0)
        x = self.embedding(interaction)
        h, _ = self.lstm(x)
        logits = self.fc(h)
        # Return logits instead of sigmoid for BCEWithLogitsLoss compatibility with autocast
        return logits

    def predict_next(self, q_seq, a_seq, target_q):
        self.eval()
        with torch.no_grad():
            if not isinstance(q_seq, torch.Tensor):
                q_seq = torch.tensor(q_seq, dtype=torch.long).unsqueeze(0)
            if not isinstance(a_seq, torch.Tensor):
                a_seq = torch.tensor(a_seq, dtype=torch.float32).unsqueeze(0)
            logits = self(q_seq, a_seq)
            preds = torch.sigmoid(logits)
            return preds[0, -1, target_q].item()


def train_dkt():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    batch_size = 256
    n_epochs = 20
    lr = 3e-4
    weight_decay = 1e-2
    patience = 3
    data_dir = "/kaggle/working/data/processed"

    print("Loading data...")
    train_data = torch.load(os.path.join(data_dir, "train.pt"), map_location="cpu")
    val_data = torch.load(os.path.join(data_dir, "val.pt"), map_location="cpu")

    n_questions = int(max(train_data["q"].max().item(), val_data["q"].max().item()))
    print(f"Num questions encoded: {n_questions}")

    train_loader = DataLoader(TensorDataset(train_data["q"], train_data["a"]), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(val_data["q"], val_data["a"]), batch_size=batch_size, shuffle=False)

    model = DKT(n_questions=n_questions).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Use BCEWithLogitsLoss which is safe for mixed precision (torch.amp.autocast)
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    scaler = GradScaler("cuda", enabled=torch.cuda.is_available())

    best_val_auc = 0.0
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0

        for q_batch, a_batch in train_loader:
            q_batch = q_batch.to(device)
            a_batch = a_batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            with autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(q_batch, a_batch)

                q_target = q_batch[:, 1:].long()
                a_target = a_batch[:, 1:]
                logits_subset = logits[:, :-1, :]
                mask = (a_target != -1) & (q_target > 0)

                gathered_logits = torch.gather(logits_subset, 2, q_target.unsqueeze(2)).squeeze(2)

                valid_logits = gathered_logits[mask]
                valid_targets = a_target[mask]
                loss = criterion(valid_logits, valid_targets).mean()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item() * q_batch.size(0)

        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        val_preds_all, val_targets_all = [], []

        with torch.no_grad():
            for q_batch, a_batch in val_loader:
                q_batch = q_batch.to(device)
                a_batch = a_batch.to(device)

                logits = model(q_batch, a_batch)
                q_target = q_batch[:, 1:].long()
                a_target = a_batch[:, 1:]
                logits_subset = logits[:, :-1, :]
                mask = (a_target != -1) & (q_target > 0)

                gathered_logits = torch.gather(logits_subset, 2, q_target.unsqueeze(2)).squeeze(2)
                
                # Apply sigmoid to logits for AUC calculation
                gathered_preds = torch.sigmoid(gathered_logits)
                
                val_preds_all.extend(gathered_preds[mask].detach().cpu().numpy())
                val_targets_all.extend(a_target[mask].detach().cpu().numpy())

        val_auc = roc_auc_score(val_targets_all, val_preds_all)
        print(f"Epoch {epoch + 1:02d}/{n_epochs} | Loss: {train_loss:.4f} | Val AUC-ROC: {val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), "/kaggle/working/dkt_best.pt")
            epochs_no_improve = 0
            print(f"  -> Saved best model with Val AUC: {best_val_auc:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  -> No improvement. Patience: {epochs_no_improve}/{patience}")
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break


train_dkt()

In [ ]:
from IPython.display import FileLink

# When the DKT model training is fully completed (3-5 hours expected), 
# this cell will generate a link to download the model checkpoint file over your browser.
FileLink(r'dkt_best.pt')